In [ ]:
# final_two_stage_rare_event.py
from __future__ import annotations
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.neighbors import NearestNeighbors


# =========================
# Repro & small utilities
# =========================
def set_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def l2norm(Z: np.ndarray) -> np.ndarray:
    Z = np.asarray(Z, dtype=np.float32)
    n = np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12
    return Z / n


def ranknorm(x: np.ndarray) -> np.ndarray:
    """Monotone rank normalization to [0,1]. Robust for blending."""
    x = np.asarray(x)
    r = np.argsort(np.argsort(x))
    return r / (len(x) - 1 + 1e-9)


# =========================
# Model: SupCon encoder
# =========================
class SupConEncoder(nn.Module):
    def __init__(self, in_dim: int, enc_dim: int = 256, proj_dim: int = 128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 512), nn.ReLU(),
            nn.Linear(512, enc_dim), nn.ReLU(),
        )
        self.proj = nn.Sequential(
            nn.Linear(enc_dim, enc_dim), nn.ReLU(),
            nn.Linear(enc_dim, proj_dim),
        )

    def forward(self, x: torch.Tensor, *, return_proj: bool = True) -> torch.Tensor:
        h = self.encoder(x)  # [B, enc_dim]
        if not return_proj:
            return h
        z = self.proj(h)     # [B, proj_dim]
        z = F.normalize(z, dim=-1)  # unit vectors for cosine
        return z


class SupConLoss(nn.Module):
    """Pull only positives (y==1) together; ignore negatives to avoid collapse."""
    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.tau = temperature

    def forward(self, z: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        z = F.normalize(z, dim=-1)
        logits = (z @ z.t()) / self.tau
        logits = logits - torch.eye(z.size(0), device=z.device) * 1e9
        y = y.view(-1, 1)
        pos_mask = (y == y.t()) & (y == 1)  # positives only
        log_denom = torch.logsumexp(logits, dim=1)
        num = torch.logsumexp(torch.where(pos_mask, logits, torch.full_like(logits, -1e9)), dim=1)
        valid = pos_mask.any(dim=1)
        loss = -(num[valid] - log_denom[valid]).mean()
        return loss


def make_label_aware_batches(y_np: np.ndarray, batch_size=2048, min_pos=128, seed=0):
    rng = np.random.default_rng(seed)
    idx_pos = np.where(y_np == 1)[0]
    idx_neg = np.where(y_np == 0)[0]
    while True:
        p = rng.choice(idx_pos, size=min_pos, replace=(len(idx_pos) < min_pos))
        n = rng.choice(idx_neg, size=batch_size - min_pos, replace=False)
        b = np.concatenate([p, n])
        rng.shuffle(b)
        yield b


def train_supcon(
    X_tr: np.ndarray, y_tr: np.ndarray, in_dim: int,
    *, epochs=10, batch_size=1024, min_pos=128,
    lr=3e-4, wd=1e-4, tau=0.07, device="cpu"
) -> SupConEncoder:
    model = SupConEncoder(in_dim=in_dim).to(device)
    crit = SupConLoss(temperature=tau)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sampler = make_label_aware_batches(y_tr, batch_size=batch_size, min_pos=min_pos, seed=0)
    steps = max(1, len(X_tr) // batch_size)

    model.train()
    for ep in range(1, epochs + 1):
        running = 0.0
        for _ in range(steps):
            b = next(sampler)
            xb = torch.from_numpy(X_tr[b]).float().to(device)
            yb = torch.from_numpy(y_tr[b]).long().to(device)
            z = model(xb, return_proj=True)
            loss = crit(z, yb)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            running += loss.item()
        print(f"[SupCon] epoch {ep:02d} loss {running/steps:.4f}")
    model.eval()
    return model


@torch.no_grad()
def compute_embeddings(model: SupConEncoder, X: np.ndarray, device="cpu", batch=65536) -> np.ndarray:
    Z = []
    for i in range(0, len(X), batch):
        xb = torch.from_numpy(X[i:i+batch]).float().to(device)
        zb = model(xb, return_proj=True).cpu().numpy()
        Z.append(zb)
    Z = np.vstack(Z).astype(np.float32)
    return l2norm(Z)  # ensure L2 norm = 1 for cosine


# =========================
# Model: MLP probe on Z
# =========================
class MLPProbe(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 256), nn.ReLU(),
            nn.Linear(256, 64), nn.ReLU(),
            nn.Linear(64, 1),
        )
    def forward(self, Z: torch.Tensor) -> torch.Tensor:
        return self.net(Z)  # [N,1]


def train_mlp_probe_on_Z(
    Z_tr: np.ndarray, y_tr: np.ndarray, *, epochs=10, lr=1e-3, wd=1e-4, device="cpu"
) -> MLPProbe:
    head = MLPProbe(Z_tr.shape[1]).to(device)
    pos_weight = torch.tensor([(y_tr == 0).sum() / max(1, (y_tr == 1).sum())], device=device)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=wd)

    Zt = torch.from_numpy(Z_tr).float().to(device)
    yt = torch.from_numpy(y_tr).float().to(device)

    head.train()
    for _ in range(epochs):
        opt.zero_grad()
        logits = head(Zt).reshape(-1)
        loss = F.binary_cross_entropy_with_logits(logits, yt, pos_weight=pos_weight)
        loss.backward(); opt.step()
    head.eval()
    return head


@torch.no_grad()
def probe_prob(head: MLPProbe, Z: np.ndarray, device="cpu") -> np.ndarray:
    logits = head(torch.from_numpy(Z).float().to(device)).reshape(-1).cpu().numpy()
    return 1.0 / (1.0 + np.exp(-logits))


# =========================
# Unsupervised density + blended rank
# =========================
def knn_density(Z_ref: np.ndarray, Z_query: np.ndarray, k: int = 20) -> np.ndarray:
    """Mean cosine similarity to k nearest neighbors in Z_ref."""
    nn = NearestNeighbors(n_neighbors=k, metric="cosine").fit(Z_ref)
    _, idx = nn.kneighbors(Z_query)
    sims = np.sum(Z_query[:, None, :] * Z_ref[idx], axis=2)  # cosine (dot) because Z are L2-normalized
    return sims.mean(axis=1)


def blended_rank_score(
    Z_ref: np.ndarray, Z_query: np.ndarray, head: MLPProbe, *,
    alpha: float = 0.5, k: int = 20, device="cpu"
) -> np.ndarray:
    """Rank-normalized blend (probe + density). For global use this can bury positives—prefer two-stage."""
    pr = probe_prob(head, Z_query, device=device)
    den = knn_density(Z_ref, Z_query, k=k)
    return alpha * ranknorm(pr) + (1.0 - alpha) * ranknorm(den)


def rerank_topK(
    Z_ref: np.ndarray, Z_query: np.ndarray, head: MLPProbe, *,
    M: int = 200, K: int = 10, alpha: float = 0.5, k: int = 20, device="cpu"
) -> np.ndarray:
    """Two-stage re-rank: (1) top-M by probe, (2) re-rank M by blended rank score, (3) return top-K indices."""
    pr = probe_prob(head, Z_query, device=device)
    idxM = np.argpartition(-pr, M)[:M]
    sM = blended_rank_score(Z_ref, Z_query[idxM], head, alpha=alpha, k=k, device=device)
    return idxM[np.argsort(-sM)[:K]]


def topk_metrics_from_indices(y: np.ndarray, idx: np.ndarray) -> tuple[float, float]:
    top = y[idx]
    purity = float(top.mean())
    hit = float((top.sum() > 0))
    return purity, hit


# =========================
# Evaluation wrapper
# =========================
def evaluate_final(
    Z_tr: np.ndarray, Z_va: np.ndarray, y_va: np.ndarray,
    Z_te: np.ndarray, y_te: np.ndarray,
    head: MLPProbe, *, alpha=0.5, k=20, M=200, K=10, device="cpu"
):
    # Global (probe only)
    pr_va = probe_prob(head, Z_va, device=device)
    pr_te = probe_prob(head, Z_te, device=device)
    va_ap, va_roc = average_precision_score(y_va, pr_va), roc_auc_score(y_va, pr_va)
    te_ap, te_roc = average_precision_score(y_te, pr_te), roc_auc_score(y_te, pr_te)

    # Top-K via two-stage re-rank
    idx_va = rerank_topK(Z_tr, Z_va, head, M=M, K=K, alpha=alpha, k=k, device=device)
    idx_te = rerank_topK(Z_tr, Z_te, head, M=M, K=K, alpha=alpha, k=k, device=device)
    pur_va, hit_va = topk_metrics_from_indices(y_va, idx_va)
    pur_te, hit_te = topk_metrics_from_indices(y_te, idx_te)

    print(f"VAL  AP={va_ap:.4f} ROC={va_roc:.4f} | purity@{K}={pur_va:.3f} P≥1={hit_va:.3f}")
    print(f"TEST AP={te_ap:.4f} ROC={te_roc:.4f} | purity@{K}={pur_te:.3f} P≥1={hit_te:.3f}")
    return (va_ap, va_roc, pur_va, hit_va), (te_ap, te_roc, pur_te, hit_te)


# =========================
# Pipeline
# =========================
def run_pipeline(
    X_np: np.ndarray, y_np: np.ndarray, *,
    test_size=0.2, val_size=0.2, seed=42, device="cpu",
    supcon_epochs=10, supcon_tau=0.07, supcon_batch=1024, supcon_min_pos=128,
    probe_epochs=10,
    alpha=0.5, k_density=20, M=200, K=10
):
    """End-to-end: split → scale → SupCon → embeddings → probe → two-stage re-rank eval."""
    set_seed(seed)

    # Split (Train → Val/Test)
    X_tr, X_te, y_tr, y_te = train_test_split(X_np, y_np, test_size=test_size, stratify=y_np, random_state=seed)
    X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=val_size, stratify=y_tr, random_state=seed)

    # Train-only scaler
    scaler = StandardScaler().fit(X_tr)
    X_tr = scaler.transform(X_tr).astype(np.float32)
    X_va = scaler.transform(X_va).astype(np.float32)
    X_te = scaler.transform(X_te).astype(np.float32)

    print(f"train pos: {int(y_tr.sum())} val pos: {int(y_va.sum())} test pos: {int(y_te.sum())}")

    # SupCon pretrain
    enc = train_supcon(
        X_tr, y_tr, in_dim=X_tr.shape[1],
        epochs=supcon_epochs, batch_size=supcon_batch, min_pos=supcon_min_pos,
        lr=3e-4, wd=1e-4, tau=supcon_tau, device=device
    )

    # Embeddings (L2-normalized)
    Z_tr = compute_embeddings(enc, X_tr, device=device)
    Z_va = compute_embeddings(enc, X_va, device=device)
    Z_te = compute_embeddings(enc, X_te, device=device)

    # Probe on Z
    head = train_mlp_probe_on_Z(Z_tr, y_tr, epochs=probe_epochs, lr=1e-3, wd=1e-4, device=device)

    # Evaluate
    return evaluate_final(
        Z_tr, Z_va, y_va, Z_te, y_te, head,
        alpha=alpha, k=k_density, M=M, K=K, device=device
    )


# =========================
# Demo / Entry point
# =========================
if __name__ == "__main__":
    # ---- Replace this block with your real X_np, y_np loader ----
    # Synthetic demo (rare ~0.7% positives)
    N, D = 30000, 20
    X_np = np.random.randn(N, D).astype(np.float32)
    y_np = (np.random.rand(N) < 0.0073).astype(np.int64)

    # Device
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Run
    run_pipeline(
        X_np, y_np,
        test_size=0.2, val_size=0.2, seed=42, device=device,
        supcon_epochs=10, supcon_tau=0.07, supcon_batch=1024, supcon_min_pos=128,
        probe_epochs=10,
        alpha=0.5, k_density=20, M=200, K=10
    )